In [6]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")

In [7]:
from langchain_community.document_loaders import TextLoader

b:\LLMOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
loader = TextLoader("B:/LLMOps/data/Agentic AI.txt", encoding="utf8")
documents = loader.load()
print(documents)

[Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='Agentic AI: Autonomous Systems That Plan, Reason, and Act\n\nIntroduction\n\nAgentic AI refers to artificial intelligence systems that can pursue\ngoals with a significant degree of autonomy. Unlike traditional AI\nmodels that simply respond to a single prompt or perform one narrowly\ndefined task, agentic systems can plan, reason, use tools, remember\nprior context, and take actions over multiple steps. They behave more\nlike capable assistants than static prediction engines. The term\n“agentic” comes from the idea of agency: the ability to make decisions\nand act intentionally in an environment to achieve an objective.\n\nThe emergence of large language models (LLMs) has accelerated the\ndevelopment of agentic AI. Models such as GPT, Claude, and Gemini can\nunderstand complex instructions, generate structured outputs, write\ncode, and interact with software tools. When combined with memory, tool\naccess, an

In [9]:
documents[0].page_content[:100]

'Agentic AI: Autonomous Systems That Plan, Reason, and Act\n\nIntroduction\n\nAgentic AI refers to artifi'

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [12]:
text_chunks=text_splitter.split_documents(documents)

In [13]:
text_chunks

[Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='Agentic AI: Autonomous Systems That Plan, Reason, and Act\n\nIntroduction'),
 Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='Agentic AI refers to artificial intelligence systems that can pursue\ngoals with a significant degree of autonomy. Unlike traditional AI'),
 Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='models that simply respond to a single prompt or perform one narrowly\ndefined task, agentic systems can plan, reason, use tools, remember'),
 Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='prior context, and take actions over multiple steps. They behave more\nlike capable assistants than static prediction engines. The term'),
 Document(metadata={'source': 'B:/LLMOps/data/Agentic AI.txt'}, page_content='“agentic” comes from the idea of agency: the ability to make decisions\nand act intentionally in an environment 

In [14]:
from langchain_community.vectorstores import FAISS

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [16]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\Jai Vadula\AppData\Local\Temp\ipykernel_3676\3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9365.53it/s]


In [17]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [18]:
vectorstore

In [19]:
retriever=vectorstore.as_retriever()

In [20]:
# Perform similarity search
query = "What is the Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Agentic AI, in contrast, is proactive. It can make a plan, gather
information, execute actions, and adjust based on outcomes. Traditional
--------------------------------------------------
Document 2:
Agentic AI refers to artificial intelligence systems that can pursue
goals with a significant degree of autonomy. Unlike traditional AI
--------------------------------------------------
Document 3:
AI answers a question; agentic AI solves the broader problem.
--------------------------------------------------
Document 4:
Architecture of an Agentic AI System

A typical agentic AI architecture includes several components.
--------------------------------------------------


In [21]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [22]:
prompt=ChatPromptTemplate.from_template(template)

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [24]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [25]:
from langchain_cohere import ChatCohere

llm_model = ChatCohere(model="command-a-03-2025")

In [26]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [29]:
rag_chain.invoke("The Future of Agentic AI")

'Agentic AI represents a significant evolution in artificial intelligence, combining reasoning, planning, memory, and tool use to solve broader problems autonomously. Unlike traditional AI, which typically answers specific questions, agentic AI systems can pursue goals with a high degree of independence. These systems are designed to plan, reason, and act, making them capable of addressing complex, multifaceted challenges. By integrating these capabilities, agentic AI moves beyond reactive responses to proactive problem-solving. This advancement holds promise for applications requiring long-term planning and decision-making. However, the context provided does not detail specific future developments or challenges for agentic AI. For a comprehensive understanding of its future, further exploration of ethical, technical, and societal implications would be necessary.'